# Step 1b — Feasibility Gate, corrected (Open TG-GATEs)

Supersedes the first version. Two defects in that run changed the numbers materially.

### Fix 1 — spontaneous lesions were being counted as toxicity
The spontaneous-lesion flag in the pathology file is named **`SP_FLG`**, which the
previous column matcher did not resolve, so the filter never applied. 39% of all
findings in the file are spontaneous (background lesions unrelated to the compound).
Among **control** animals the effect is stark: 323 of 327 liver findings are
spontaneous. Leaving them in makes control animals look injured and inflates the
day-4 exclusion count.

### Fix 2 — the 22.8% "match rate" was not a failure
The pathology file is a **findings list, not an examination record**: an animal with
normal histopathology has no row at all. This is confirmed by the first version's own
output — under `any_finding`, every one of the 4,068 animals in the file was positive.
So 22.8% is simply the share of arrays with a finding, and treating a missing record
as "no finding" is correct. Disregard the earlier warning about that number.

The encoding fix you added (cp932) was right — the file is Shift-JIS, not UTF-8.

### New label rule
`union` = **an injurious lesion at any grade, OR any finding graded moderate or worse.**

This separates injury from adaptation on toxicologic grounds. Minimal or slight
hypertrophy — the single most common finding, and an adaptive response rather than
damage — is excluded, while moderate or severe hypertrophy still counts, as does
necrosis at any grade.

## 1. Setup

In [ ]:
import io, json, urllib.request, zipfile
from pathlib import Path
import pandas as pd

BASE = "https://dbarchive.biosciencedbc.jp/data/open-tggates/LATEST"
FILES = {"pathology": "open_tggates_pathology.zip",
         "attribute": "open_tggates_cel_file_attribute.zip"}

EARLY_DAY      = 4
LATE_DAYS      = (15, 29)
EARLY_MAX_RATE = 0.0    # arm is pre-injury only if no day-4 animal has a finding
LATE_MIN_RATE  = 0.5    # arm is positive if >= half the animals have a finding later

KEY = ["EXP_ID", "GROUP_ID", "INDIVIDUAL_ID"]

# Finding types treated as injury rather than adaptation.
INJURY_PATTERN = r"necrosis|degenerat|fibrosis|hemorrhage|cholestasis"

OUT = Path("/content/tggates_annot"); OUT.mkdir(parents=True, exist_ok=True)
print("output:", OUT, "| pandas", pd.__version__)

## 2. Download and read (Shift-JIS encoded)

In [ ]:
def fetch(name):
    dest = OUT / FILES[name]
    if dest.exists() and dest.stat().st_size:
        print(f"{dest.name} present ({dest.stat().st_size/1024:.0f} KB)"); return dest
    req = urllib.request.Request(f"{BASE}/{FILES[name]}",
                                 headers={"User-Agent": "research-script/1.0"})
    with urllib.request.urlopen(req, timeout=180) as r:
        dest.write_bytes(r.read())
    print(f"saved {dest.name} ({dest.stat().st_size/1024:.0f} KB)"); return dest

def read_zip_table(path):
    with zipfile.ZipFile(path) as z:
        member = max([n for n in z.namelist()
                      if n.lower().endswith((".csv", ".tsv", ".txt"))
                      and not n.startswith("__MACOSX")],
                     key=lambda n: z.getinfo(n).file_size)
        blob = z.read(member)
    for sep in ("\t", ","):
        for enc in ("cp932", "shift_jis", "utf-8", "latin1"):
            try:
                df = pd.read_csv(io.BytesIO(blob), sep=sep, encoding=enc, low_memory=False)
                if df.shape[1] > 2:
                    print(f"  {member}: {df.shape} (sep={sep!r}, enc={enc})")
                    return df
            except Exception:
                pass
    raise RuntimeError(f"could not parse {path.name}")

path_df = read_zip_table(fetch("pathology"))
attr_df = read_zip_table(fetch("attribute"))
assert "SP_FLG" in path_df.columns, "SP_FLG missing -- check the pathology file layout"
print("\nSP_FLG:", dict(path_df.SP_FLG.value_counts()))
print("GRADE_TYPE:", dict(path_df.GRADE_TYPE.value_counts()))

## 3. Sanity check — does "no record" really mean "no finding"?

If every animal was examined and the file lists only findings, control animals should
appear almost exclusively with *spontaneous* lesions. That is the check below; if it
holds, filling missing records with "no finding" is sound.

In [ ]:
ctl = path_df[(path_df.ORGAN == "Liver") & (path_df.DOSE_LEVEL == "Control")]
n_sp = int(ctl.SP_FLG.astype(bool).sum())
print(f"control liver findings : {len(ctl)}")
print(f"  spontaneous          : {n_sp}")
print(f"  non-spontaneous      : {len(ctl) - n_sp}")
print(f"  control animals with any record: "
      f"{ctl.groupby(KEY).ngroups}")
print("\nExpected: nearly all spontaneous. If many are non-spontaneous, the")
print("control arms carry compound-unrelated injury and the design needs rethinking.")

print("\n--- most common non-spontaneous liver findings ---")
print(path_df[(path_df.ORGAN == "Liver") & (~path_df.SP_FLG.astype(bool))]
      .FINDING_TYPE.value_counts().head(15).to_string())

## 4. Label rules

In [ ]:
RULES = {
    "any_finding":     dict(injury=False, min_grade=None),
    "moderate_plus":   dict(injury=False, min_grade={"moderate", "severe"}),
    "injury":          dict(injury=True,  min_grade=None),
    "injury_moderate": dict(injury=True,  min_grade={"moderate", "severe"}),
    "union":           dict(union=True),   # recommended
}

def animal_labels(rule):
    cfg = RULES[rule]
    df = path_df[path_df.ORGAN == "Liver"].copy()
    not_spont = ~df.SP_FLG.astype(bool)                       # FIX 1
    is_injury = df.FINDING_TYPE.astype(str).str.contains(
        INJURY_PATTERN, case=False, na=False)
    is_mod = df.GRADE_TYPE.astype(str).str.strip().str.lower().isin({"moderate", "severe"})

    if cfg.get("union"):
        hit = not_spont & (is_injury | is_mod)
    else:
        hit = not_spont.copy()
        if cfg["injury"]:
            hit &= is_injury
        if cfg["min_grade"]:
            hit &= df.GRADE_TYPE.astype(str).str.strip().str.lower().isin(cfg["min_grade"])

    df["_f"] = hit
    out = df.groupby(KEY)["_f"].max().reset_index().rename(columns={"_f": "finding"})
    out["key"] = out[KEY].astype(str).agg("_".join, axis=1)
    return out

## 5. Arm inventory and classification

In [ ]:
def inventory(labels):
    a = attr_df[(attr_df.ORGAN == "Liver")
                & attr_df.SINGLE_REPEAT_TYPE.astype(str).str.contains("repeat", case=False)
                & attr_df.SPECIES.astype(str).str.contains("rat", case=False)].copy()
    a["key"] = a[KEY].astype(str).agg("_".join, axis=1)
    per = a.SACRIFICE_PERIOD.astype(str)
    a["day"] = (per.str.extract(r"(\d+)")[0].astype(float)
                   .where(per.str.contains("day", case=False, na=False)))
    a = a.dropna(subset=["day"]); a["day"] = a["day"].astype(int)

    m = a.merge(labels[["key", "finding"]], on="key", how="left")
    # a missing pathology record means no finding -- see section 3
    m["finding"] = m["finding"].fillna(False).astype(bool)

    inv = (m.groupby(["COMPOUND_NAME", "DOSE_LEVEL", "day"])
             .agg(n=("finding", "size"), n_finding=("finding", "sum")).reset_index())
    inv["rate"] = inv.n_finding / inv.n
    return inv, m

def classify(inv):
    rows = []
    for (c, d), g in inv[inv.DOSE_LEVEL != "Control"].groupby(["COMPOUND_NAME", "DOSE_LEVEL"]):
        by_day = g.set_index("day")["rate"].to_dict()
        n_day  = g.set_index("day")["n"].to_dict()
        early  = by_day.get(EARLY_DAY)
        late   = [by_day[x] for x in LATE_DAYS if x in by_day]
        r = {"compound": c, "dose_level": d, "early_rate": early,
             "n_early": n_day.get(EARLY_DAY),
             "late_rate_max": max(late) if late else None, "outcome": None}
        if early is None:                 r["status"] = "no_early_timepoint"
        elif not late:                    r["status"] = "no_late_timepoint"
        elif early > EARLY_MAX_RATE:      r["status"] = "already_injured_at_early"
        else:
            r["status"] = "admitted"
            r["outcome"] = int(max(late) >= LATE_MIN_RATE)
        rows.append(r)
    return pd.DataFrame(rows)

## 6. Verdict across all rules

In [ ]:
summary = {}
print(f"{'rule':17s} {'admitted':>9s} {'pos':>5s} {'neg':>5s} {'cmpd':>5s} "
      f"{'cmpd+':>6s} {'day4 n':>7s} {'pos%':>6s}")
print("-" * 70)
for rule in RULES:
    lab = animal_labels(rule)
    inv, merged = inventory(lab)
    arms = classify(inv)
    adm = arms[arms.status == "admitted"]
    pos = adm[adm.outcome == 1]
    s = {"admitted": len(adm), "positive": len(pos), "negative": len(adm) - len(pos),
         "compounds": int(adm.compound.nunique()),
         "compounds_with_positive": int(pos.compound.nunique()),
         "day4_samples": int(adm.n_early.fillna(0).sum()),
         "positive_pct": round(100 * len(pos) / max(len(adm), 1), 1),
         "excluded_injured_at_day4": int((arms.status == "already_injured_at_early").sum())}
    summary[rule] = s
    print(f"{rule:17s} {s['admitted']:9d} {s['positive']:5d} {s['negative']:5d} "
          f"{s['compounds']:5d} {s['compounds_with_positive']:6d} "
          f"{s['day4_samples']:7d} {s['positive_pct']:5.1f}%")
    arms.to_csv(OUT / f"arms_{rule}.csv", index=False)
    inv.to_csv(OUT / f"inventory_{rule}.csv", index=False)
    merged[["BARCODE", "key", "COMPOUND_NAME", "DOSE_LEVEL", "day", "finding"]].to_csv(
        OUT / f"array_labels_{rule}.csv", index=False)

(OUT / "feasibility_summary_corrected.json").write_text(json.dumps(summary, indent=2))
print("\ncmpd+ = compounds contributing at least one positive arm (LOCO fold viability)")

## 7. The recommended rule in detail

In [ ]:
arms = pd.read_csv(OUT / "arms_union.csv")
adm = arms[arms.status == "admitted"]
pos = adm[adm.outcome == 1]

print(f"admitted arms {len(adm)} | positive {len(pos)} ({100*len(pos)/len(adm):.1f}%) "
      f"| compounds {adm.compound.nunique()} | with positives {pos.compound.nunique()}")
print("\npositive arms per compound:")
print(pos.compound.value_counts().to_string())
print("\nall positive arms:")
print(pos[["compound", "dose_level", "late_rate_max", "n_early"]].to_string(index=False))

## 8. Download outputs

In [ ]:
import shutil
arch = shutil.make_archive("/content/step1b_outputs", "zip", OUT)
print("archive:", arch)
try:
    from google.colab import files; files.download(arch)
except Exception as e:
    print("files are in", OUT, "|", e)

## Interpreting the trade-off

The label rule sets the balance between toxicologic defensibility and statistical
power, and the two pull in opposite directions:

- `any_finding` gives the most positives but counts minimal hypertrophy — adaptation,
  not injury — as hepatotoxicity. A toxicology reviewer will contest it.
- `injury` is clean but leaves very few positive arms, concentrated in a handful of
  compounds, so most leave-one-compound-out folds contain no positive at all.
- `union` sits between them on a stated principle rather than on convenience.

Pick the rule on toxicologic grounds and state that reasoning in Methods. Report the
others as a sensitivity analysis — showing the conclusion survives the choice is far
stronger than defending one definition. Do **not** pick whichever rule maximises the
arm count.